In [1]:
import sys
sys.path.insert(0, '..')

from helper.golbal_variables import * 
from ast_dataset import Fast_AST_Dataset, fast_collate_fn
from ast_model import ASTGenreClassifier

z:\dlgen-AI-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch 

device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = 10   # genres
model = ASTGenreClassifier(num_classes).to(device)

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from torch.utils.data import WeightedRandomSampler, DataLoader, Subset
from collections import Counter
import numpy as np
from pathlib import Path

dataset = Fast_AST_Dataset(
    data_dir=Path(r"C:\Users\satwi\OneDrive\Desktop\dlgen-ai-project\genres_pt")
)

unique_song_ids = list(set([item[0] for item in dataset.index_map]))
num_songs = len(unique_song_ids)

np.random.seed(42)
np.random.shuffle(unique_song_ids)

train_song_count = int(0.8 * num_songs)
train_songs = set(unique_song_ids[:train_song_count])
val_songs = set(unique_song_ids[train_song_count:])

print(f"\nSongs split: {len(train_songs)} train, {len(val_songs)} val")

# Map chunks to the designated train/val songs
train_indices = [i for i, (song_idx, start, aug_ver) in enumerate(dataset.index_map) 
                 if song_idx in train_songs]
val_indices = [i for i, (song_idx, start, aug_ver) in enumerate(dataset.index_map) 
               if song_idx in val_songs]

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

print(f"Train chunks: {len(train_dataset)}")
print(f"Val chunks: {len(val_dataset)}")


# Get labels for training chunks
train_labels = [dataset.labels[i] for i in train_indices]

class_counts = Counter(train_labels)

class_weights = {cls: 1.0 / count for cls, count in class_counts.items()}
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

print("\nClass distribution in training chunks:")
genre_names = {v: k for k, v in dataset.class_map.items()}
for cls, count in sorted(class_counts.items()):
    print(f"  {genre_names[cls]}: {count} chunks (weight: {class_weights[cls]:.5f})")


train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    sampler=sampler,
    collate_fn=fast_collate_fn,
    num_workers=0,
    pin_memory=True if device == "cuda" else False
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=fast_collate_fn,
    num_workers=0,
    pin_memory=True if device == "cuda" else False
)

print(f"\n{'='*70}")
print(f"DATALOADERS READY")
print(f"{'='*70}")
print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches: {len(val_dataloader)}")
print(f"{'='*70}\n")

# Safety Check
train_song_set = set(dataset.index_map[i][0] for i in train_indices)
val_song_set = set(dataset.index_map[i][0] for i in val_indices)

overlap = train_song_set & val_song_set
assert len(overlap) == 0, f"⚠ Data leakage! {len(overlap)} songs in both train and val"
print(f"✅ No data leakage - train and val songs are completely separate")

Loaded 3000 pre-computed spectrogram chunks.

Songs split: 800 train, 200 val
Train chunks: 2400
Val chunks: 600

Class distribution in training chunks:
  blues: 228 chunks (weight: 0.00439)
  classical: 249 chunks (weight: 0.00402)
  country: 240 chunks (weight: 0.00417)
  disco: 246 chunks (weight: 0.00407)
  hiphop: 234 chunks (weight: 0.00427)
  jazz: 234 chunks (weight: 0.00427)
  metal: 252 chunks (weight: 0.00397)
  pop: 219 chunks (weight: 0.00457)
  reggae: 231 chunks (weight: 0.00433)
  rock: 267 chunks (weight: 0.00375)

DATALOADERS READY
Train batches: 150
Val batches: 38

✅ No data leakage - train and val songs are completely separate


In [ ]:
from utils import train_ast_model 

train_ast_model(
    model=model,
    device=device,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    epochs=20
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\satwi\_netrc.



AST GENRE CLASSIFICATION TRAINING



wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


STAGE 1: Training only classifier head (lr=1e-3)
        Trainable: 9,226 / 86,196,490

Epoch [  1/15] | Train Loss: 0.7104 | Train Acc:  77.79% | Val Loss: 0.5153 | Val Acc:  84.17% | LR: 1.00e-03 | Time: 134.7s


wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


        ✓ Saved best model (val_acc: 84.17%, val_loss: 0.5153)
Epoch [  2/15] | Train Loss: 0.3814 | Train Acc:  87.92% | Val Loss: 0.5216 | Val Acc:  81.50% | LR: 1.00e-03 | Time: 139.9s
Epoch [  3/15] | Train Loss: 0.3114 | Train Acc:  90.21% | Val Loss: 0.4809 | Val Acc:  83.83% | LR: 1.00e-03 | Time: 142.3s
Epoch [  4/15] | Train Loss: 0.2782 | Train Acc:  91.92% | Val Loss: 0.5002 | Val Acc:  82.17% | LR: 1.00e-03 | Time: 139.4s
Epoch [  5/15] | Train Loss: 0.2191 | Train Acc:  93.50% | Val Loss: 0.4568 | Val Acc:  85.00% | LR: 1.00e-03 | Time: 139.2s
        ✓ Saved best model (val_acc: 85.00%, val_loss: 0.4568)

STAGE 2: Unfreezing top 2 AST encoder layers (lr=5e-5)
        Trainable: 14,184,970 / 86,196,490

Epoch [  6/15] | Train Loss: 0.2355 | Train Acc:  92.46% | Val Loss: 0.5844 | Val Acc:  84.17% | LR: 5.00e-05 | Time: 181.7s
Epoch [  7/15] | Train Loss: 0.1276 | Train Acc:  96.00% | Val Loss: 0.4828 | Val Acc:  85.67% | LR: 5.00e-05 | Time: 181.3s
        ✓ Saved best mod

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


Epoch [ 15/15] | Train Loss: 0.0041 | Train Acc:  99.88% | Val Loss: 0.7713 | Val Acc:  86.50% | LR: 5.00e-06 | Time: 3493.2s

AST TRAINING COMPLETE
Total Training Time    : 315.5 minutes
Best Validation Acc    : 87.33%
Best Validation Loss   : 0.5874
Model saved as         : best_ast_genre.pth



epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
epoch_time,▁▁▁▁▁▁▁▁▁▁█████
learning_rate,█████▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▄▅▅▆▆▇████████
train_loss,█▅▄▄▃▃▂▁▁▁▁▁▁▁▁
val_accuracy,▄▁▄▂▅▄▆▆▇▇██▄▁▇
val_loss,▂▂▁▁▁▂▁▃▃▂▂▄▅█▄
epoch,15
epoch_time,3493.22574
learning_rate,1e-05
stage,full_unfreeze


87.33333333333333

In [6]:
import torch 
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ASTGenreClassifier(num_classes)
model.load_state_dict(torch.load("best_ast_genre.pth", map_location=device))
model = model.to(device)

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\visha\AppData\Local\Temp\ipykernel_18460\3364953039.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default

In [8]:
import pandas as pd
from predict import predict_ast

genre_map = {
    'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4,
    'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9
}

test_df = pd.read_csv(MESSY_DATA_DIR / "test.csv")
output = {}
for idx, row in test_df.iterrows():
    song_id = row["id"]
    filename = row["filename"]
    audio_path = MESSY_DATA_DIR / filename

    predicted_genre, confidence, all_probs = predict_ast(
        model=model,
        audio_path=audio_path,
        device=device,
        genre_map=genre_map
    )

    output[song_id] = predicted_genre
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_df)} songs...")

output_df = pd.DataFrame(list(output.items()), columns=['id', 'genre'])
output_df.set_index("id", inplace=True)
output_df.to_csv("output 2.csv")

Processed 100/3020 songs...
Processed 200/3020 songs...
Processed 300/3020 songs...
Processed 400/3020 songs...
Processed 500/3020 songs...
Processed 600/3020 songs...
Processed 700/3020 songs...
Processed 800/3020 songs...
Processed 900/3020 songs...
Processed 1000/3020 songs...
Processed 1100/3020 songs...
Processed 1200/3020 songs...
Processed 1300/3020 songs...
Processed 1400/3020 songs...
Processed 1500/3020 songs...
Processed 1600/3020 songs...
Processed 1700/3020 songs...
Processed 1800/3020 songs...
Processed 1900/3020 songs...
Processed 2000/3020 songs...
Processed 2100/3020 songs...
Processed 2200/3020 songs...
Processed 2300/3020 songs...
Processed 2400/3020 songs...
Processed 2500/3020 songs...
Processed 2600/3020 songs...
Processed 2700/3020 songs...
Processed 2800/3020 songs...
Processed 2900/3020 songs...
Processed 3000/3020 songs...
